In [1]:
import pandas as pd
import pandapower as pp
import pandapower.plotting as plot
import pandas as pd
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from utils import run_time_series

Create Grid (Copy from other notebook)

In [2]:
# create empty net
net = pp.create_empty_network()

# create buses 0..17 (bus 0 is 20 kV, the rest are 0.4 kV)
buses = [pp.create_bus(net, vn_kv=20.0 if i==0 else 0.4, name=f"Bus {i}") for i in range(18)]

# create bus elements (use the first few buses as examples)
pp.create_ext_grid(net, bus=buses[0], vm_pu=1.02, name="Grid Connection")
for i in range(1, 18):
    pp.create_load(net, bus=buses[i], p_mw=0.1, q_mvar=0.005, name=f"Load {i}") # pyright: ignore[reportArgumentType]
    pp.create_sgen(net, bus=buses[i], p_mw=0.1, q_mvar=0.005, name=f"Sgen {i}") # type: ignore


# create branch elements
tid = pp.create_transformer(
    net, hv_bus=buses[0], lv_bus=buses[1], std_type="0.4 MVA 20/0.4 kV", name="Trafo"
)
# create primary branch: buses 1 -> 2 -> ... -> 8
for i in range(1, 8):
    pp.create_line(net, from_bus=buses[i], to_bus=buses[i+1], length_km=0.100, name=f"Line {i}-{i+1}", std_type="NAYY 4x50 SE")

# create second branch: connect bus 1 to bus 9, then 9 -> 10 -> ... -> 17
pp.create_line(net, from_bus=buses[1], to_bus=buses[9], length_km=0.100, name="Line 1-9", std_type="NAYY 4x50 SE")
for i in range(9, 17):
    pp.create_line(net, from_bus=buses[i], to_bus=buses[i+1], length_km=0.100, name=f"Line {i}-{i+1}", std_type="NAYY 4x50 SE")

In [3]:
your_gen_data = pd.read_csv("../Data/GenerationData_B.csv", index_col=0)
# your_gen_data = pd.read_csv("../Data/GenerationData_C.csv", index_col=0)
your_load_data = pd.read_csv("../Data/LoadData_B.csv", index_col=0)
# your_load_data = pd.read_csv("../Data/LoadData_C.csv", index_col=0)

your_net = net

# Copy order from other notebook (Warning: Different gen and load order swapped)
your_gen_order = [13, 14, 5, 7, 9, 10, 3, 12, 15, 11, 2, 16, 1, 0, 8, 6, 4]
your_load_order = [13, 11, 1, 4, 7, 8, 10, 2, 9, 14, 3, 16, 15, 0, 6, 12, 5]

In [4]:
# save your_gen and your_load in single data frame and save to csv
df_save = pd.DataFrame({'generation_order':your_gen_order,'load_order':your_load_order})

In [5]:
Group_Letter = 'B' # change this to your group letter
df_save.to_csv(f'Group_{Group_Letter}_Assignment_2_Data_Set_C_order.csv',index=False)

In [6]:
# load again to check if it is correct
df = pd.read_csv(f'Group_{Group_Letter}_Assignment_2_Data_Set_C_order.csv')

In [7]:
your_gen_order_loaded = df['generation_order'].values
your_load_order_loaded = df['load_order'].values

In [8]:
your_gen_order == your_gen_order_loaded
your_load_order == your_load_order_loaded

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True])

In [9]:
res_ext, res_lines = run_time_series(your_gen_data, your_load_data, your_net, your_gen_order, your_load_order, results_suffix="evaluation")

  0%|          | 0/24 [00:00<?, ?it/s]

100%|██████████| 24/24 [00:01<00:00, 17.62it/s]


In [10]:
all(res_lines.max(axis=1) <= 100.0) # has to be True

True

In [11]:
res_lines.max(axis=1).sum() # to be minimized

79.72359777620204